# Notebook de Simulação e Teste para o LiveTrader e o SetupScanner

Este notebook permite executar o fluxo de trabalho de dois componentes principais:
1. **LiveTrader (Passo a Passo)**: Para um único ativo escolhido, ideal para depuração detalhada.
2. **SetupScanner (Execução Completa)**: Para todos os ativos habilitados, ideal para obter uma visão geral das oportunidades atuais.

**Pré-requisitos:**
1. O terminal MetaTrader 5 deve estar aberto e logado.
2. Os modelos de produção devem ter sido gerados pelo script `train_model.py`.

In [ ]:
import pandas as pd
import yaml
import sys
from pathlib import Path
import MetaTrader5 as mt5
import time
from datetime import datetime, timezone
import logging

logging.basicConfig(level=logging.INFO)

# Adiciona a pasta 'src' ao path para permitir as importações dos nossos módulos
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.live_trader import LiveTrader
from src.setup_scanner import SetupScanner

## Parte 1: Teste Assistido do LiveTrader (Ativo Único)
---
### Passo 1.1: Escolha o Ativo e Inicialize o Trader

**Ação:** Defina a variável `DATA_TICKER_PARA_TESTAR` com o ticker de DADOS HISTÓRICOS (ex: "WDO$").

In [ ]:
# --- ESCOLHA O ATIVO AQUI (use o ticker de dados históricos) ---
DATA_TICKER_PARA_TESTAR = "WDO$"
# ------------------------------------------------------------

trader = LiveTrader(config_path='configs/main.yaml')
is_initialized = trader.initialize()
asset_state_para_testar = None

if is_initialized and DATA_TICKER_PARA_TESTAR in trader.asset_states:
    print(f"Trader inicializado com sucesso. Foco do teste: {DATA_TICKER_PARA_TESTAR}")
    asset_state_para_testar = trader.asset_states[DATA_TICKER_PARA_TESTAR]
else:
    print(f"Falha ao inicializar ou ticker '{DATA_TICKER_PARA_TESTAR}' não encontrado/carregado.")

### Passo 1.2: Executar um Único Ciclo de Decisão (Single Tick)

In [ ]:
def run_single_tick(trader_instance, asset_state):
    if not asset_state:
        print("Estado do ativo não inicializado.")
        return

    config = asset_state['config']
    data_ticker = config['ticker']
    live_config = config['live_trading']
    order_ticker = live_config.get('ticker_order', data_ticker)
    timeframe_str = live_config['timeframe_str']
    mt5_timeframe = trader_instance._get_mt5_timeframe_from_string(timeframe_str)
    
    print(f"--- Executando ciclo de decisão para: {data_ticker} ({timeframe_str}) ---")

    # 1. Buscar dados recentes
    print(f"Buscando candles de {data_ticker}...")
    latest_data = trader_instance.provider.get_latest_rates(data_ticker, 300, mt5_timeframe)

    if latest_data.empty:
        print("Não foi possível obter dados recentes.")
        return
    
    # --- ALTERAÇÃO AQUI: Captura a data/hora do último candle --- 
    last_candle_time = latest_data.index[-1].strftime('%Y-%m-%d %H:%M:%S')
    print(f"Último candle recebido: {last_candle_time}")
    display(latest_data.tail(3))

    # 2. Gerar features
    featured_data = asset_state["strategy"].define_features(latest_data)
    X_live = featured_data[asset_state["strategy"].get_feature_names()].dropna()
    if X_live.empty:
        print("Dados insuficientes para gerar features.")
        return

    # 3. Gerar sinal
    print("\nGerando sinal com o modelo de IA...")
    signal = asset_state["model"].predict(X_live)[-1]
    signal_text = 'COMPRA' if signal == 1 else 'VENDA'

    # Lógica de fallback para o preço sugerido
    suggested_price = 0.0
    price_source = "N/A"
    symbol_info = mt5.symbol_info_tick(order_ticker)

    if symbol_info and symbol_info.ask > 0 and symbol_info.bid > 0:
        suggested_price = symbol_info.ask if signal == 1 else symbol_info.bid
        price_source = "Tick (Tempo Real)"
    elif not latest_data.empty:
        suggested_price = latest_data['close'].iloc[-1]
        price_source = "Fechamento do Último Candle"

    if suggested_price > 0:
        # --- ALTERAÇÃO AQUI: Adiciona a data/hora do candle na mensagem --- 
        print(f"==> SINAL GERADO: {signal_text} ({signal}) @ Preço Sugerido: ${suggested_price:.2f} (Ref. Candle: {last_candle_time}, Fonte: {price_source}) ==<")
    else:
        print(f"==> SINAL GERADO: {signal_text} ({signal}) (Não foi possível obter preço) ==<")
    
    # 4. Lógica de decisão
    if asset_state["position"] is None:
        if signal == 1: trader_instance._execute_trade(data_ticker, 'BUY')
        elif signal == 0: trader_instance._execute_trade(data_ticker, 'SELL')
    else:
        print(f"Posição já aberta para {data_ticker} ({asset_state['position']}).")
        
    print("--- Ciclo de decisão concluído ---")

### Passo 1.3: Executar o Teste de Ciclo Único

Execute esta célula para rodar a simulação do `LiveTrader` para o ativo selecionado.

In [ ]:
print("\n--- Iniciando Teste de Ciclo Único para o LiveTrader ---")
try:
    if is_initialized and asset_state_para_testar:
        run_single_tick(trader, asset_state_para_testar)
        print("Teste concluído.")
except Exception as e:
    print(f"Erro durante o teste do LiveTrader: {e}")

## Parte 2: Teste do Scanner de Setups (Todos os Ativos)
---
### Passo 2.1: Executar o Scanner

Esta célula irá instanciar o `SetupScanner` e executá-lo uma vez para **todos os ativos habilitados** no `main.yaml`, apresentando as sugestões que atenderem tanto ao sinal da IA quanto às regras de setup configuradas.

In [ ]:
print("\n--- Iniciando Teste do Scanner de Setups ---")
try:
    # O scanner se conecta e desconecta do MT5 internamente.
    scanner = SetupScanner(config_path='configs/main.yaml')
    scanner.scan_all_assets()
except Exception as e:
    print(f"Erro durante a execução do scanner: {e}")

## Parte 3: Encerrar a Conexão
---

Ao final de todos os seus testes, execute esta célula para garantir que a conexão com o MetaTrader 5 seja encerrada corretamente.

In [ ]:
print("Encerrando conexão com o MetaTrader 5...")
mt5.shutdown()
print("Conexão encerrada.")